## 1. Check GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > GPU")

## 2. Install Dependencies

In [ ]:
%%bash
# Install system dependencies
apt-get update -qq
apt-get install -y git wget

In [ ]:
# Install Python packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.45.2 timm==1.0.9 accelerate==1.2.1 diffusers==0.31.0
!pip install -q websockets opencv-python pillow numpy fvcore
!pip install -q flash-attn --no-build-isolation

print("✅ Dependencies installed")

## 3. Clone Evo-1 Repository

In [ ]:
import os
from pathlib import Path

# Clone Evo-1 repository
if not Path("/content/Evo-1").exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

os.chdir("/content/Evo-1")

## 4. Download Model Checkpoint

In [ ]:
from huggingface_hub import snapshot_download
import os

# Create checkpoint directory
checkpoint_dir = "/content/Evo-1/checkpoints/libero"
os.makedirs(checkpoint_dir, exist_ok=True)

# Download checkpoint from HuggingFace
print("Downloading Evo-1 LIBERO checkpoint (1.4GB)...")
snapshot_download(
    repo_id="MINT-SJTU/Evo1_LIBERO",
    local_dir=checkpoint_dir,
    local_dir_use_symlinks=False
)

print("✅ Checkpoint downloaded")
print("\nCheckpoint files:")
!ls -lh {checkpoint_dir}

## 5. Configure Model for CUDA

In [ ]:
import json

# Update config to use CUDA
config_path = "/content/Evo-1/checkpoints/libero/config.json"
with open(config_path, 'r') as f:
    config = json.load(f)

config['device'] = 'cuda'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Config updated to use CUDA")
print(f"Device: {config['device']}")

## 6. Fix Server Code for Multi-Device Support

In [ ]:
# Fix Evo1_server.py to support dynamic device selection
server_file = "/content/Evo-1/Evo_1/scripts/Evo1_server.py"

# Read the file
with open(server_file, 'r') as f:
    content = f.read()

# Fix 1: Update load_model_and_normalizer to use device from config
content = content.replace(
    'model = model.to("cuda")',
    'device = config.get("device", "cuda")\n    model = model.to(device)'
)

# Fix 2: Update decode_image_from_list signature
content = content.replace(
    'def decode_image_from_list(img_list):',
    'def decode_image_from_list(img_list, device="cuda"):'
)

# Fix 3: Update decode_image_from_list to use device parameter
content = content.replace(
    'return transforms.ToTensor()(pil).to("cuda")',
    'return transforms.ToTensor()(pil).to(device)'
)

# Fix 4: Update infer_from_json_dict
content = content.replace(
    'def infer_from_json_dict(data: dict, model, normalizer):\n    device = "cuda"',
    'def infer_from_json_dict(data: dict, model, normalizer):\n    device = next(model.parameters()).device'
)

# Fix 5: Update image decoding call
content = content.replace(
    'images = [decode_image_from_list(img) for img in data["image"]]',
    'images = [decode_image_from_list(img, device=device) for img in data["image"]]'
)

# Write back
with open(server_file, 'w') as f:
    f.write(content)

print("✅ Server code fixed for device support")

In [ ]:
# Fix InternVL3 embedder for device-specific configuration
embedder_file = "/content/Evo-1/Evo_1/model/internvl3/internvl3_embedder.py"

with open(embedder_file, 'r') as f:
    content = f.read()

# Replace hardcoded torch_dtype and use_flash_attn with conditional logic
old_code = '''        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            use_flash_attn=True,
            low_cpu_mem_usage=True,
            _fast_init=False,
        ).to(self.device)'''

new_code = '''        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
        
        # Use appropriate dtype and features based on device
        if device == "cuda":
            torch_dtype = torch.bfloat16
            use_flash_attn = True
        else:
            # MPS/CPU: use float32 and disable flash attention
            torch_dtype = torch.float32
            use_flash_attn = False
        
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            trust_remote_code=True,
            use_flash_attn=use_flash_attn,
            low_cpu_mem_usage=True,
            _fast_init=False,
        ).to(self.device)'''

content = content.replace(old_code, new_code)

with open(embedder_file, 'w') as f:
    f.write(content)

print("✅ Embedder code fixed for device support")

## 7. Setup Ngrok Tunnel (For Remote Access)

Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Install ngrok
!pip install -q pyngrok

from pyngrok import ngrok
import getpass

# Get auth token (you only need to do this once per session)
print("Get your auth token from: https://dashboard.ngrok.com/get-started/your-authtoken")
ngrok_token = getpass.getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(ngrok_token)

print("✅ Ngrok configured")

## 8. Start Evo-1 Server

This will:
1. Load the 1B parameter InternVL3 model (~2-3 minutes)
2. Start WebSocket server on port 9000
3. Create ngrok tunnel for remote access

In [ ]:
import os
import sys
import threading
import time
from pyngrok import ngrok

# Change to scripts directory
os.chdir("/content/Evo-1/Evo_1/scripts")

# Update checkpoint path in server
with open("Evo1_server.py", 'r') as f:
    server_content = f.read()

server_content = server_content.replace(
    'ckpt_dir = "/Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark/models/Evo-1/checkpoints/libero"',
    'ckpt_dir = "/content/Evo-1/checkpoints/libero"'
)

with open("Evo1_server.py", 'w') as f:
    f.write(server_content)

print("🚀 Starting Evo-1 server...")
print("Loading model (this will take 2-3 minutes)...\n")

# Start server in background thread
def run_server():
    os.system("python Evo1_server.py")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(60)

# Create ngrok tunnel
public_url = ngrok.connect(9000, "tcp")
print("\n" + "="*60)
print("✅ Server is running!")
print("="*60)
print(f"\n🌐 Public URL: {public_url}")
print("\n📝 Update your local client to use this URL:")
print(f"   SERVER_URL = '{public_url}'")
print("\n⚠️  Keep this cell running! The server will stop if you interrupt it.")
print("="*60)

# Keep the cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nServer stopped")

## 9. Test Server Connection (Optional)

In [ ]:
import asyncio
import websockets
import json
import numpy as np

async def test_connection():
    # Test data
    test_data = {
        "image": [np.random.randint(0, 255, (480, 640, 3)).tolist() for _ in range(3)],
        "state": np.zeros(24).tolist(),
        "prompt": "test prompt",
        "image_mask": [1, 1, 1],
        "action_mask": 1
    }
    
    try:
        async with websockets.connect("ws://localhost:9000") as ws:
            await ws.send(json.dumps(test_data))
            response = await ws.recv()
            print("✅ Server connection successful!")
            print(f"Response shape: {np.array(json.loads(response)).shape}")
    except Exception as e:
        print(f"❌ Connection failed: {e}")

# Run test
await test_connection()

## 10. Monitor GPU Usage

In [ ]:
!nvidia-smi

## Next Steps

1. **Copy the ngrok URL** from cell 8
2. **On your local machine**, update the client script:
   ```python
   SERVER_URL = "tcp://X.tcp.ngrok.io:XXXXX"  # Use your ngrok URL
   ```
3. **Run the LIBERO client** from your Mac:
   ```bash
   cd /Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark
   ./scripts/run_libero_client.sh
   ```

4. **Expected runtime**: 2-3 hours for full evaluation
5. **Expected success rate**: ~94.8% (from paper)

---

**Tips:**
- Keep this notebook running during evaluation
- Monitor GPU usage in cell 10
- Colab will disconnect after ~12 hours (even with Pro)
- Save checkpoint if you need to resume